In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import math
import operator
import csv

In [2]:
pd.options.display.max_colwidth = 250

In [3]:
from sentence_transformers import SentenceTransformer
 
model = SentenceTransformer('LazarusNLP/all-indo-e5-small-v4')

In [4]:
# Membaca file madaniyah.csv untuk mendapatkan nomor surah
madaniyah_chapters = []
with open('../madaniyah.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Melewati header
    for row in csv_reader:
        madaniyah_chapters.append(int(row[1]))
        

sentences = []

for chapter in madaniyah_chapters:

  google_translate = pd.read_csv('../data/google_translate/chapter_' + str(chapter) + '.csv')
  kemenag = pd.read_csv('../data/kemenag/chapter_' + str(chapter) + '.csv')
  king_fahd = pd.read_csv('../data/king_fahd/chapter_' + str(chapter) + '.csv')

  sentence1 = google_translate['google_translate'].tolist()
  sentence2 = kemenag['kemenag'].tolist()
  sentence3 = king_fahd['king_fahd'].tolist()

  sentences.append(sentence1)
  sentences.append(sentence2)
  sentences.append(sentence3)


sentence_new = []
for i in range(0, len(sentences)):
  for j in range(0, len(sentences[i])):
    sentence_new.append(sentences[i][j])

sentence_embeddings = model.encode(sentence_new)

In [5]:
sentence_embeddings.shape

(3897, 384)

In [6]:
# Inisialisasi
index_map = []
current_index = 0
df = pd.DataFrame(columns=[
    'Chapter', 'Verse', 'Google Translate', 'Kemenag', 'King Fahd',
    'Fahd - Google', 'Kemenag - Google', 'Fahd - Kemenag'
])

# Bangun peta indeks untuk masing-masing surah
for chapter in madaniyah_chapters:
    df_chapter = pd.read_csv(f'../data/google_translate/chapter_{chapter}.csv')

    n_verses = len(df_chapter)

    index_map.append({
        'chapter': chapter,
        'start_index': current_index,
        'num_verses': n_verses
    })

    current_index += 3 * n_verses  # 3 versi: king_fahd, kemenag, google

# Loop seluruh surah dan ayat untuk hitung similarity
for mapping in index_map:
    chapter = mapping['chapter']
    start = mapping['start_index']
    n_verses = mapping['num_verses']

    for verse in range(n_verses):
        i_google = start + verse
        i_kemenag = start + n_verses + verse
        i_fahd = start + 2 * n_verses + verse

        # Hitung cosine similarity
        sim_fahd = cosine_similarity([sentence_embeddings[i_fahd]], [sentence_embeddings[i_google]])[0][0]
        sim_kemenag = cosine_similarity([sentence_embeddings[i_kemenag]], [sentence_embeddings[i_google]])[0][0]
        sim_fahd_kemenag = cosine_similarity([sentence_embeddings[i_fahd]], [sentence_embeddings[i_kemenag]])[0][0]

        dict_row = {
            'Chapter': chapter,
            'Verse': verse + 1,
            'Google Translate': sentence_new[i_google],
            'Kemenag': sentence_new[i_kemenag],
            'King Fahd': sentence_new[i_fahd],
            'Fahd - Google': sim_fahd,
            'Kemenag - Google': sim_kemenag,
            'Fahd - Kemenag': sim_fahd_kemenag
        }

        df = pd.concat([df, pd.DataFrame([dict_row])], ignore_index=True)

# Tampilkan hasil akhir
df


/var/folders/p8/hpcxrgfd4wv_j3fp535z1ytc0000gn/T/ipykernel_93284/2195464410.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([dict_row])], ignore_index=True)


,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000
1,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244
2,2,3,"Orang-orang yang beriman kepada yang gaib, mendirikan shalat, dan menafkahkan sebagian rezeki yang Kami berikan kepada mereka.","(Yaitu) mereka yang beriman kepada yang gaib, melaksanakan salat, dan menginfakkan sebagian rezeki yang Kami berikan kepada mereka,","(yaitu) mereka yang beriman kepada yang gaib, yang mendirikan salat, dan menafkahkan sebagian rezeki yang Kami anugerahkan kepada mereka,",0.901631,0.898958,0.975567
3,2,4,"Dan orang-orang yang beriman kepada kitab yang diturunkan kepadamu dan kitab yang diturunkan sebelum kamu, dan kepada kehidupan akhirat, mereka adalah yakin.","dan mereka beriman kepada (Alquran) yang diturunkan kepadamu (Muhammad) dan (kitab-kitab) yang telah diturunkan sebelum engkau, dan mereka yakin akan adanya akhirat.","dan mereka yang beriman kepada Kitab (Al-Qur`ān) yang telah diturunkan kepadamu dan kitab-kitab yang telah diturunkan sebelummu, serta mereka yakin akan adanya (kehidupan) akhirat.",0.909239,0.857523,0.920318
4,2,5,Mereka itulah yang mendapat petunjuk dari Tuhannya dan mereka itulah orang-orang yang beruntung.,"Merekalah yang mendapat petunjuk dari Tuhannya, dan mereka itulah orang-orang yang beruntung.","Mereka itulah yang tetap mendapat petunjuk dari Tuhan mereka, dan merekalah orang-orang yang beruntung.",0.931456,0.992417,0.932781
...,...,...,...,...,...,...,...,...
1294,66,8,"Wahai orang-orang yang beriman, bertobatlah kepada Allah dengan taubat yang tulus. Mudah-mudahan Tuhanmu akan menghapus kesalahan-kesalahanmu dan memasukkanmu ke dalam surga yang mengalir di bawahnya sungai-sungai, pada hari ketika Allah tidak ak...","Wahai orang-orang yang beriman! Bertobatlah kepada Allah dengan tobat yang semurni-murninya, mudah-mudahan Tuhan kamu akan menghapus kesalahan-kesalahanmu dan memasukkan kamu ke dalam surga-surga yang mengalir di bawahnya sungai-sungai, pada hari...","Hai orang-orang yang beriman, bertobatlah kepada Allah dengan tobat yang semurni-murninya, mudah-mudahan Tuhan kamu akan menghapus kesalahan-kesalahanmu dan memasukkanmu ke dalam surga yang mengalir di bawahnya sungai-sungai, pada hari ketika All...",0.907043,0.875991,0.928247
1295,66,9,"Wahai Nabi, perangilah orang-orang kafir dan munafik, dan bersikaplah keras terhadap mereka. Tempat kembali mereka adalah neraka, dan seburuk-buruk tempat kembali.",Wahai Nabi! Perangilah orang-orang kafir dan orang-orang munafik dan bersikap keraslah terhadap mereka. Tempat mereka adalah neraka Jahanam dan itulah seburuk-buruk tempat kembali.,"Hai Nabi, perangilah orang-orang kafir dan orang-orang munafik dan bersikap keraslah terhadap mereka. Tempat mereka adalah jahanam dan itu adalah seburuk-buruknya tempat kembali.",0.913609,0.960408,0.928341
1296,66,10,"Allah menjadikan istri Nuh dan istri Luth sebagai perumpamaan bagi orang-orang yang kafir. Keduanya berada di bawah pengawasan dua hamba Kami yang saleh, lalu keduanya mengkhianati keduanya, maka keduanya tidak dapat menolong mereka sedikit pun ...","Allah membuat perumpamaan bagi orang-orang kafir, istri Nuh dan istri Luṭ. Keduanya berada di bawah pengawasan dua orang hamba yang saleh di antara hamba-hamba Kami; lalu kedua istri itu berkhianat kepada kedua suaminya, tetapi kedua suaminya itu...","Allah membuat istri Nūḥ dan istri Lūṭ sebagai perumpamaan bagi orang-orang kafir. Keduanya berada di bawah pengawasan dua orang hamba yang saleh di antara hamba-hamba Kami; lalu kedua istri itu berkhianat kepada kedua suaminya, maka kedua suamin...",0.904372,0.954068,0.930484
1297,66,11,"Dan Allah menjadikan perumpamaan bagi orang-or

In [7]:
df.to_csv('../Semantic Analysis Results/cosine_similarity.csv', index=False)

## BLEU SCORE

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [ ]:
bleu_kemenag_google = []
bleu_fahd_google = []
bleu_fahd_kemenag = []

# Smoothing function membantu menghindari skor 0 untuk kalimat pendek
chencherry = SmoothingFunction()

# Iterasi melalui setiap baris DataFrame untuk menghitung skor
for index, row in df.iterrows():
    # Tokenisasi (memecah kalimat menjadi kata-kata)
    # Referensi harus dalam format list of lists: [[kata1, kata2, ...]]
    ref_google = [row['Google Translate'].split()]
    ref_kemenag = [row['Kemenag'].split()]
    
    # Kandidat adalah list kata-kata: [kata1, kata2, ...]
    candidate_kemenag = row['Kemenag'].split()
    candidate_fahd = row['King Fahd'].split()

    # Hitung skor BLEU untuk Kemenag (kandidat) vs Google (referensi)
    score_kg = sentence_bleu(ref_google, candidate_kemenag, smoothing_function=chencherry.method1)
    bleu_kemenag_google.append(score_kg)

    # Hitung skor BLEU untuk King Fahd (kandidat) vs Google (referensi)
    score_fg = sentence_bleu(ref_google, candidate_fahd, smoothing_function=chencherry.method1)
    bleu_fahd_google.append(score_fg)
    
    # Hitung skor BLEU untuk King Fahd (kandidat) vs Kemenag (referensi)
    score_fk = sentence_bleu(ref_kemenag, candidate_fahd, smoothing_function=chencherry.method1)
    bleu_fahd_kemenag.append(score_fk)

# Tambahkan kolom baru berisi skor BLEU ke DataFrame
df['BLEU Kemenag-Google'] = bleu_kemenag_google
df['BLEU Fahd-Google'] = bleu_fahd_google
df['BLEU Fahd-Kemenag'] = bleu_fahd_kemenag

# Tampilkan DataFrame dengan kolom skor semantik dan skor BLEU
# Anda dapat menampilkan seluruh df atau hanya beberapa kolom untuk verifikasi
pd.set_option('display.max_rows', 10) # Mengatur tampilan agar tidak terlalu panjang
display(df[[
    'Chapter', 'Verse', 
    'Fahd - Google', 'BLEU Fahd-Google',
    'Kemenag - Google', 'BLEU Kemenag-Google',
    'Fahd - Kemenag', 'BLEU Fahd-Kemenag'
]])

### Menemukan Perbedaan Maksimal

In [8]:
maximum_difference_fahd_google = df.loc[(df['Fahd - Google'] < 0.5)]

maximum_difference_fahd_google

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000
286,3,1,nyeri,Alif lām Mīm.,Alif Lām Mīm.,0.010567,0.125248,0.824753
339,3,54,"Dan mereka berencana, dan Allah pun berencana. Dan Allah adalah sebaik-baik perencana.","Dan mereka (orang-orang kafir) membuat tipu daya, maka Allah pun membalas tipu daya. Dan Allah sebaik-baik pembalas tipu daya.","Orang-orang kafir itu membuat tipu daya, dan Allah membalas tipu daya mereka itu. Dan Allah sebaik-baik pembalas tipu daya.",0.447102,0.476180,0.945183
583,4,98,"Kecuali orang-orang tertindas di antara laki-laki, perempuan, dan anak-anak yang tidak dapat merancang rencana dan juga tidak ditunjukkan jalan.","Kecuali mereka yang tertindas baik laki-laki atau perempuan dan anak-anak yang tidak berdaya dan tidak mengetahui jalan (untuk berhijrah),","kecuali mereka yang tertindas, baik laki-laki atau wanita ataupun anak-anak yang tidak mampu berdaya upaya dan tidak mengetahui jalan (untuk hijrah),",0.493049,0.612253,0.825672
818,9,37,"Interkalasi hanyalah penambahan kekufuran yang dengannya orang-orang kafir disesatkan. Mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain agar mereka dapat menyempurnakan apa yang diharamkan Allah, dan dengan demikia...","Sesungguhnya pengunduran (bulan haram) itu hanya menambah kekafiran. Orang-orang kafir disesatkan dengan (pengunduran) itu, mereka menghalalkannya satu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menyesuaikan dengan bilangan...","Sesungguhnya mengundur-undurkan bulan haram itu adalah menambah kekafiran, disesatkan orang-orang yang kafir dengan mengundur-undurkan itu, mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menses...",0.346780,0.354384,0.879055
959,24,49,"Dan jika kebenaran itu milik mereka, mereka akan menerimanya dengan berserah diri.","Tetapi, jika kebenaran dipihak mereka, mereka datang kepadanya (Rasul) dengan patuh.","Tetapi jika keputusan itu untuk (kemaslahatan) mereka, mereka datang kepada rasul dengan patuh.",0.314125,0.711123,0.525156
988,33,14,"Dan jika surga itu datang kepada mereka dari daerahnya, kemudian mereka diminta untuk mengujinya, niscaya mereka akan mengujinya, dan mereka tidak akan tinggal di dalamnya kecuali sebentar saja.","Dan kalau (Yasrib) diserang dari segala penjuru, dan mereka diminta agar membuat kekacauan, niscaya mereka mengerjakannya; dan hanya sebentar saja mereka menunggu.","Kalau (Yaṡrib) diserang dari segala penjuru, kemudian diminta kepada mereka supaya murtad , niscaya mereka mengerjakannya dan mereka tiada akan menunda untuk murtad itu, melainkan dalam waktu yang singkat.",0.438173,0.428026,0.772614


In [9]:
maximum_difference_kemenag_google = df.loc[(df['Kemenag - Google'] < 0.5)]

maximum_difference_kemenag_google

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000
286,3,1,nyeri,Alif lām Mīm.,Alif Lām Mīm.,0.010567,0.125248,0.824753
339,3,54,"Dan mereka berencana, dan Allah pun berencana. Dan Allah adalah sebaik-baik perencana.","Dan mereka (orang-orang kafir) membuat tipu daya, maka Allah pun membalas tipu daya. Dan Allah sebaik-baik pembalas tipu daya.","Orang-orang kafir itu membuat tipu daya, dan Allah membalas tipu daya mereka itu. Dan Allah sebaik-baik pembalas tipu daya.",0.447102,0.476180,0.945183
818,9,37,"Interkalasi hanyalah penambahan kekufuran yang dengannya orang-orang kafir disesatkan. Mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain agar mereka dapat menyempurnakan apa yang diharamkan Allah, dan dengan demikia...","Sesungguhnya pengunduran (bulan haram) itu hanya menambah kekafiran. Orang-orang kafir disesatkan dengan (pengunduran) itu, mereka menghalalkannya satu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menyesuaikan dengan bilangan...","Sesungguhnya mengundur-undurkan bulan haram itu adalah menambah kekafiran, disesatkan orang-orang yang kafir dengan mengundur-undurkan itu, mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menses...",0.346780,0.354384,0.879055
918,24,8,"Dan mereka menangkal azab itu dengan memberi kesaksian empat kali atas nama Allah, bahwa sesungguhnya dia termasuk orang-orang yang berdusta.","Dan istri itu terhindar dari hukuman apabila dia bersumpah empat kali atas (nama) Allah bahwa dia (suaminya) benar-benar termasuk orang-orang yang berdusta,",Istrinya itu dihindarkan dari hukuman oleh sumpahnya empat kali atas nama Allah; sesungguhnya suaminya itu benar-benar termasuk orang-orang yang dusta.,0.542857,0.490454,0.891828
988,33,14,"Dan jika surga itu datang kepada mereka dari daerahnya, kemudian mereka diminta untuk mengujinya, niscaya mereka akan mengujinya, dan mereka tidak akan tinggal di dalamnya kecuali sebentar saja.","Dan kalau (Yasrib) diserang dari segala penjuru, dan mereka diminta agar membuat kekacauan, niscaya mereka mengerjakannya; dan hanya sebentar saja mereka menunggu.","Kalau (Yaṡrib) diserang dari segala penjuru, kemudian diminta kepada mereka supaya murtad , niscaya mereka mengerjakannya dan mereka tiada akan menunda untuk murtad itu, melainkan dalam waktu yang singkat.",0.438173,0.428026,0.772614


In [10]:
maximum_difference = df.loc[(df['Fahd - Google'] < 0.5) & (df['Kemenag - Google'] < 0.5)]

maximum_difference

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000
286,3,1,nyeri,Alif lām Mīm.,Alif Lām Mīm.,0.010567,0.125248,0.824753
339,3,54,"Dan mereka berencana, dan Allah pun berencana. Dan Allah adalah sebaik-baik perencana.","Dan mereka (orang-orang kafir) membuat tipu daya, maka Allah pun membalas tipu daya. Dan Allah sebaik-baik pembalas tipu daya.","Orang-orang kafir itu membuat tipu daya, dan Allah membalas tipu daya mereka itu. Dan Allah sebaik-baik pembalas tipu daya.",0.447102,0.476180,0.945183
818,9,37,"Interkalasi hanyalah penambahan kekufuran yang dengannya orang-orang kafir disesatkan. Mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain agar mereka dapat menyempurnakan apa yang diharamkan Allah, dan dengan demikia...","Sesungguhnya pengunduran (bulan haram) itu hanya menambah kekafiran. Orang-orang kafir disesatkan dengan (pengunduran) itu, mereka menghalalkannya satu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menyesuaikan dengan bilangan...","Sesungguhnya mengundur-undurkan bulan haram itu adalah menambah kekafiran, disesatkan orang-orang yang kafir dengan mengundur-undurkan itu, mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menses...",0.346780,0.354384,0.879055
988,33,14,"Dan jika surga itu datang kepada mereka dari daerahnya, kemudian mereka diminta untuk mengujinya, niscaya mereka akan mengujinya, dan mereka tidak akan tinggal di dalamnya kecuali sebentar saja.","Dan kalau (Yasrib) diserang dari segala penjuru, dan mereka diminta agar membuat kekacauan, niscaya mereka mengerjakannya; dan hanya sebentar saja mereka menunggu.","Kalau (Yaṡrib) diserang dari segala penjuru, kemudian diminta kepada mereka supaya murtad , niscaya mereka mengerjakannya dan mereka tiada akan menunda untuk murtad itu, melainkan dalam waktu yang singkat.",0.438173,0.428026,0.772614


peneliti menyadari pada chapter 2 dan 3 ayat 1 terdapat nilai cosine yang bernilai 0.
setelah dicari lebih lanjut, menurut para ahli, ayat-ayat tersebut memang tidak memiliki arti sehingga penerjemahannya ke bahasa indonesia pun tidak bisa diketahui dan para ahli sepakat untuk menerjemahkannya sesuai suara ayat tersebut. 
ini adalah apa

Mengapa Ayat Ini Tidak Layak Dianalisis secara Semantik:
1. Tidak Memiliki Makna Linguistik yang Diketahui Ulama dan ahli tafsir sepakat bahwa huruf-huruf muqattaʿat ini tidak diketahui maknanya secara pasti, dan hanya Allah SWT yang mengetahui maksudnya. Maka, tidak ada padanan semantik yang bisa diukur, baik oleh manusia maupun mesin.
2. Diterjemahkan Sebagai Pelafalan Bunyi, Bukan Makna Karena tidak memiliki arti semantik, para penerjemah Al-Qur’an memilih menuliskan huruf-huruf tersebut sesuai pengucapannya, seperti “Alif Lam Mim”, dan bukan menerjemahkannya dalam bentuk kata bermakna.
3. Google Translate Menerjemahkan Secara Salah dan Kontekstual Tidak Sah Google Translate, yang bekerja berdasarkan model data umum, menyalahartikan huruf-huruf tersebut menjadi kata-kata yang tidak relevan, seperti "nyeri", karena tidak memahami konteks wahyu. Ini menghasilkan nilai cosine yang rendah secara teknis, tetapi bukan karena ketidakcocokan makna, melainkan karena metode yang tidak sesuai.

In [15]:
cleaning = pd.read_csv('../Semantic Analysis Results/cosine_similarity.csv')

cleaning = cleaning.drop([0, 286])
cleaning = cleaning.reset_index(drop=True)
cleaning.head()

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244
1,2,3,"Orang-orang yang beriman kepada yang gaib, mendirikan shalat, dan menafkahkan sebagian rezeki yang Kami berikan kepada mereka.","(Yaitu) mereka yang beriman kepada yang gaib, melaksanakan salat, dan menginfakkan sebagian rezeki yang Kami berikan kepada mereka,","(yaitu) mereka yang beriman kepada yang gaib, yang mendirikan salat, dan menafkahkan sebagian rezeki yang Kami anugerahkan kepada mereka,",0.901631,0.898958,0.975567
2,2,4,"Dan orang-orang yang beriman kepada kitab yang diturunkan kepadamu dan kitab yang diturunkan sebelum kamu, dan kepada kehidupan akhirat, mereka adalah yakin.","dan mereka beriman kepada (Alquran) yang diturunkan kepadamu (Muhammad) dan (kitab-kitab) yang telah diturunkan sebelum engkau, dan mereka yakin akan adanya akhirat.","dan mereka yang beriman kepada Kitab (Al-Qur`ān) yang telah diturunkan kepadamu dan kitab-kitab yang telah diturunkan sebelummu, serta mereka yakin akan adanya (kehidupan) akhirat.",0.909239,0.857523,0.920318
3,2,5,Mereka itulah yang mendapat petunjuk dari Tuhannya dan mereka itulah orang-orang yang beruntung.,"Merekalah yang mendapat petunjuk dari Tuhannya, dan mereka itulah orang-orang yang beruntung.","Mereka itulah yang tetap mendapat petunjuk dari Tuhan mereka, dan merekalah orang-orang yang beruntung.",0.931456,0.992417,0.932781
4,2,6,"Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, engkau (Muhammad) beri peringatan atau tidak engkau beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.",1.000000,0.977054,0.977054


In [16]:
cleaning.to_csv('../Semantic Analysis Results/cosine_similarity.csv')

In [17]:
maximum_difference_fahd_kemenang = df.loc[(df['Fahd - Kemenag'] < 0.5)]
maximum_difference_fahd_kemenang

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag


In [18]:
print("jumlah ayat semantik rendah Fahd - Google: ", len(maximum_difference_fahd_google))
print("jumlah ayat semantik rendah Kemenag - Google: ", len(maximum_difference_kemenag_google))
print("jumlah ayat semantik rendah di kedua: ", len(maximum_difference))

jumlah ayat semantik rendah Fahd - Google:  7
jumlah ayat semantik rendah Kemenag - Google:  6
jumlah ayat semantik rendah di kedua:  5


In [19]:
df['total'] = df['Fahd - Google'] + df['Kemenag - Google']
df.head(2)

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000,0.021134
1,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244,1.861109


In [20]:
df['max sum'] = df.groupby(['Chapter'])['total'].transform('max')
df.head(2)

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total,max sum
0,2,1,nyeri,Alif Lām Mīm.,Alif Lām Mīm.,0.010567,0.010567,1.000000,0.021134,1.977054
1,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244,1.861109,1.977054


In [21]:
df_max_sum = df.loc[(df['max sum'] == df['total'])]
df_max_sum

,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total,max sum
5,2,6,"Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, engkau (Muhammad) beri peringatan atau tidak engkau beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.",1.000000,0.977054,0.977054,1.977054,1.977054
416,3,131,Dan peliharalah dirimu dari api neraka yang disediakan untuk orang-orang kafir.,"Dan peliharalah dirimu dari api neraka, yang telah disiapkan untuk orang-orang kafir.","Dan peliharalah dirimu dari api neraka, yang disediakan untuk orang-orang yang kafir.",0.992948,0.988078,0.987602,1.981026,1.981026
553,4,68,Dan Kami tunjukkan mereka kepada jalan yang lurus.,dan pasti Kami tunjukan kepada mereka jalan yang lurus.,dan pasti Kami tunjuki mereka kepada jalan yang lurus.,0.980154,0.954838,0.970797,1.934992,1.934992
747,5,86,"Dan orang-orang yang kafir dan mendustakan ayat-ayat Kami, mereka itu adalah penghuni neraka.","Dan orang-orang yang kafir serta mendustakan ayat-ayat Kami, mereka itulah penghuni neraka.","Dan orang-orang kafir serta mendustakan ayat-ayat Kami, mereka itulah penghuni neraka.",0.989377,0.992759,0.995986,1.982136,1.982136
870,9,89,"Allah telah menyediakan bagi mereka surga-surga yang mengalir di bawahnya sungai-sungai, dan mereka kekal di dalamnya. Itulah kemenangan yang agung.","Allah telah menyediakan bagi mereka surga yang mengalir di bawahnya sungai-sungai, mereka kekal di dalamnya. Itulah kemenangan yang agung.","Allah telah menyediakan bagi mereka surga yang mengalir di bawahnya sungai-sungai, mereka kekal di dalamnya. Itulah kemenangan yang besar.",0.959270,0.988579,0.973153,1.947849,1.947849
944,24,34,"Dan sungguh, Kami telah turunkan kepadamu ayat-ayat yang menjelaskan kepadamu, dan contoh dari orang-orang terdahulu sebelum kamu, dan pelajaran bagi orang-orang yang bertakwa.","Dan sungguh, Kami telah menurunkan kepada kamu ayat-ayat yang memberi penjelasan, dan contoh-contoh dari orang-orang yang terdahulu sebelum kamu dan sebagai pelajaran bagi orang-orang yang bertakwa.",Dan sesungguhnya Kami telah menurunkan kepada kamu ayat-ayat yang memberi penerangan dan contoh-contoh dari orang-orang yang terdahulu sebelum kamu dan pelajaran bagi orang-orang yang bertakwa.,0.956758,0.973387,0.957617,1.930145,1.930145
1016,33,42,"Dan bertasbihlah kepada-Nya, pagi dan petang.",dan bertasbihlah kepada-Nya pada waktu pagi dan petang.,Dan bertasbihlah kepada-Nya pada waktu pagi dan petang.,0.985886,0.984013,0.996785,1.969899,1.969899
1054,47,7,"Hai orang-orang yang beriman, jika kamu menolong Allah, niscaya Dia akan menolong kamu dan meneguhkan kedudukanmu.","Wahai orang-orang yang beriman! Jika kamu menolong (agama) Allah, niscaya Dia akan menolongmu dan meneguhkan kedudukanmu.","Hai orang-orang yang beriman, jika kamu menolong (agama) Allah, niscaya Dia akan menolongmu dan meneguhkan kedudukanmu.",0.983254,0.946193,0.952714,1.929447,1.929447
1086,48,1,Sesungguhnya Kami telah memberikan kemenangan yang nyata kepadamu.,"Sungguh, Kami telah memberikan kepadamu kemenangan yang nyata.","Sesungguhnya Kami telah memberikan kepadamu kemenangan yang nyata,",0.976276,0.968684,0.953683,1.944959,1.944959
1132,49,18,Sesungguhnya Allah mengetahui apa yang gaib di langit dan di bumi. Dan Allah Maha Melihat apa yang kamu kerjakan.,"Sungguh, Allah mengetahui apa yang gaib di langit dan di bumi. Dan Allah Maha Melihat apa yang kamu kerjakan.",Sesungguhnya Allah mengetahui apa yang gaib di langit dan bumi. Dan Allah Maha Melihat apa yang kamu kerjakan.,0.998519,0.987239,0.986964,1.985758,1.985758


In [ ]:
# df_max_sum.to_csv('../Semantic Analysis Results/max_sum_in_every_chapter.csv')

In [23]:
df = pd.read_csv('../Semantic Analysis Results/cosine_similarity.csv')

In [24]:
print('Fahd - Google mean - ', df['Fahd - Google'].mean())
print('Fahd - Google std - ', df['Fahd - Google'].std())
print('\n')

print('Kemenag -Google mean - ', df['Kemenag - Google'].mean())
print('Kemenag - Google std - ', df['Kemenag - Google'].std())
print('\n')

print('Fahd - Kemeag mean- ', df['Fahd - Kemenag'].mean())
print('Fahd - Kemeag std- ', df['Fahd - Kemenag'].std())

Fahd - Google mean -  0.8238365265844256
Fahd - Google std -  0.09974022608540699


Kemenag -Google mean -  0.835603050146492
Kemenag - Google std -  0.09048852411476206


Fahd - Kemeag mean-  0.8936215573400155
Fahd - Kemeag std-  0.07098203306125421


In [27]:
df['total'] = df['Fahd - Google'] + df['Kemenag - Google']
df.head()

,Unnamed: 0,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total,min sum
0,0,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244,1.861109,1.831757
1,1,2,3,"Orang-orang yang beriman kepada yang gaib, mendirikan shalat, dan menafkahkan sebagian rezeki yang Kami berikan kepada mereka.","(Yaitu) mereka yang beriman kepada yang gaib, melaksanakan salat, dan menginfakkan sebagian rezeki yang Kami berikan kepada mereka,","(yaitu) mereka yang beriman kepada yang gaib, yang mendirikan salat, dan menafkahkan sebagian rezeki yang Kami anugerahkan kepada mereka,",0.901631,0.898958,0.975567,1.800589,1.831757
2,2,2,4,"Dan orang-orang yang beriman kepada kitab yang diturunkan kepadamu dan kitab yang diturunkan sebelum kamu, dan kepada kehidupan akhirat, mereka adalah yakin.","dan mereka beriman kepada (Alquran) yang diturunkan kepadamu (Muhammad) dan (kitab-kitab) yang telah diturunkan sebelum engkau, dan mereka yakin akan adanya akhirat.","dan mereka yang beriman kepada Kitab (Al-Qur`ān) yang telah diturunkan kepadamu dan kitab-kitab yang telah diturunkan sebelummu, serta mereka yakin akan adanya (kehidupan) akhirat.",0.909239,0.857523,0.920318,1.766763,1.831757
3,3,2,5,Mereka itulah yang mendapat petunjuk dari Tuhannya dan mereka itulah orang-orang yang beruntung.,"Merekalah yang mendapat petunjuk dari Tuhannya, dan mereka itulah orang-orang yang beruntung.","Mereka itulah yang tetap mendapat petunjuk dari Tuhan mereka, dan merekalah orang-orang yang beruntung.",0.931456,0.992417,0.932781,1.923873,1.831757
4,4,2,6,"Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, engkau (Muhammad) beri peringatan atau tidak engkau beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.",1.000000,0.977054,0.977054,1.977054,1.831757


In [28]:
df['min sum'] = df.groupby(['Chapter'])['total'].transform('min')

df.head()

,Unnamed: 0,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total,min sum
0,0,2,2,Kitab ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa.,"Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,",0.915161,0.945948,0.962244,1.861109,1.013135
1,1,2,3,"Orang-orang yang beriman kepada yang gaib, mendirikan shalat, dan menafkahkan sebagian rezeki yang Kami berikan kepada mereka.","(Yaitu) mereka yang beriman kepada yang gaib, melaksanakan salat, dan menginfakkan sebagian rezeki yang Kami berikan kepada mereka,","(yaitu) mereka yang beriman kepada yang gaib, yang mendirikan salat, dan menafkahkan sebagian rezeki yang Kami anugerahkan kepada mereka,",0.901631,0.898958,0.975567,1.800589,1.013135
2,2,2,4,"Dan orang-orang yang beriman kepada kitab yang diturunkan kepadamu dan kitab yang diturunkan sebelum kamu, dan kepada kehidupan akhirat, mereka adalah yakin.","dan mereka beriman kepada (Alquran) yang diturunkan kepadamu (Muhammad) dan (kitab-kitab) yang telah diturunkan sebelum engkau, dan mereka yakin akan adanya akhirat.","dan mereka yang beriman kepada Kitab (Al-Qur`ān) yang telah diturunkan kepadamu dan kitab-kitab yang telah diturunkan sebelummu, serta mereka yakin akan adanya (kehidupan) akhirat.",0.909239,0.857523,0.920318,1.766763,1.013135
3,3,2,5,Mereka itulah yang mendapat petunjuk dari Tuhannya dan mereka itulah orang-orang yang beruntung.,"Merekalah yang mendapat petunjuk dari Tuhannya, dan mereka itulah orang-orang yang beruntung.","Mereka itulah yang tetap mendapat petunjuk dari Tuhan mereka, dan merekalah orang-orang yang beruntung.",0.931456,0.992417,0.932781,1.923873,1.013135
4,4,2,6,"Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, engkau (Muhammad) beri peringatan atau tidak engkau beri peringatan, mereka tidak akan beriman.","Sesungguhnya orang-orang kafir, sama saja bagi mereka, kamu beri peringatan atau tidak kamu beri peringatan, mereka tidak akan beriman.",1.000000,0.977054,0.977054,1.977054,1.013135


In [29]:
df_least_sum = df.loc[(df['min sum'] == df['total'])]

df_least_sum.to_csv('../Semantic Analysis Results/least_sum_in_every_chapter.csv')

In [30]:
df_least_sum

,Unnamed: 0,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag,total,min sum
136,136,2,138,"Pewarna Allah, dan siapakah yang lebih baik dari Allah dalam hal pewarnaan? Dan kita adalah hamba-hamba-Nya.","""Ṣibgah Allah"" Siapa yang lebih baik ṣibgah-nya daripada Allah? Dan kepada-Nya kami menyembah.",Ṣibgah Allah. Dan siapakah yang lebih baik Ṣibgah-nya daripada Allah? Dan hanya kepada-Nya-lah kami menyembah.,0.509156,0.503979,0.962761,1.013135,1.013135
337,337,3,54,"Dan mereka berencana, dan Allah pun berencana. Dan Allah adalah sebaik-baik perencana.","Dan mereka (orang-orang kafir) membuat tipu daya, maka Allah pun membalas tipu daya. Dan Allah sebaik-baik pembalas tipu daya.","Orang-orang kafir itu membuat tipu daya, dan Allah membalas tipu daya mereka itu. Dan Allah sebaik-baik pembalas tipu daya.",0.447102,0.476180,0.945183,0.923282,0.923282
581,581,4,98,"Kecuali orang-orang tertindas di antara laki-laki, perempuan, dan anak-anak yang tidak dapat merancang rencana dan juga tidak ditunjukkan jalan.","Kecuali mereka yang tertindas baik laki-laki atau perempuan dan anak-anak yang tidak berdaya dan tidak mengetahui jalan (untuk berhijrah),","kecuali mereka yang tertindas, baik laki-laki atau wanita ataupun anak-anak yang tidak mampu berdaya upaya dan tidak mengetahui jalan (untuk hijrah),",0.493049,0.612253,0.825672,1.105302,1.105302
762,762,5,103,"Allah tidak menjadikan sesuatu pun yang Bahiroh, Sa'ibah, Wasilah dan Hami, akan tetapi orang-orang yang kafir itu mengada-adakan kebohongan terhadap Allah, dan kebanyakan mereka tidak dapat berpikir.","Allah tidak pernah mensyariatkan adanya Baḥīrah, Sāibah, Waṣīlah dan Ḥām. Tetapi orang-orang kafir membuat-buat kedustaan terhadap Allah, dan kebanyakan mereka tidak mengerti.","Allah sekali-kali tidak pernah mensyariatkan adanya baḥīrah, sā`ibah, waṣīlah, dan ḥām. Akan tetapi, orang-orang kafir membuat-buat kedustaan terhadap Allah, dan kebanyakan mereka tidak mengerti.",0.609727,0.614196,0.964871,1.223923,1.223923
816,816,9,37,"Interkalasi hanyalah penambahan kekufuran yang dengannya orang-orang kafir disesatkan. Mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain agar mereka dapat menyempurnakan apa yang diharamkan Allah, dan dengan demikia...","Sesungguhnya pengunduran (bulan haram) itu hanya menambah kekafiran. Orang-orang kafir disesatkan dengan (pengunduran) itu, mereka menghalalkannya satu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menyesuaikan dengan bilangan...","Sesungguhnya mengundur-undurkan bulan haram itu adalah menambah kekafiran, disesatkan orang-orang yang kafir dengan mengundur-undurkan itu, mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menses...",0.346780,0.354384,0.879055,0.701164,0.701164
957,957,24,49,"Dan jika kebenaran itu milik mereka, mereka akan menerimanya dengan berserah diri.","Tetapi, jika kebenaran dipihak mereka, mereka datang kepadanya (Rasul) dengan patuh.","Tetapi jika keputusan itu untuk (kemaslahatan) mereka, mereka datang kepada rasul dengan patuh.",0.314125,0.711123,0.525156,1.025248,1.025248
986,986,33,14,"Dan jika surga itu datang kepada mereka dari daerahnya, kemudian mereka diminta untuk mengujinya, niscaya mereka akan mengujinya, dan mereka tidak akan tinggal di dalamnya kecuali sebentar saja.","Dan kalau (Yasrib) diserang dari segala penjuru, dan mereka diminta agar membuat kekacauan, niscaya mereka mengerjakannya; dan hanya sebentar saja mereka menunggu.","Kalau (Yaṡrib) diserang dari segala penjuru, kemudian diminta kepada mereka supaya murtad , niscaya mereka mengerjakannya dan mereka tiada akan menunda untuk murtad itu, melainkan dalam waktu yang singkat.",0.438173,0.428026,0.772614,0.866199,0.866199
1050,1050,47,5,Dia akan membimbing mereka dan mengatur urusan mereka dengan benar.,"Allah akan memberi petunjuk kepada mereka dan memperbaiki keadaan mereka,","Allah akan memberi pimpinan 

In [33]:
df = pd.read_csv('../Semantic Analysis Results/cosine_similarity.csv')
maximum_difference = df.loc[(df['Fahd - Google']<0.5) & (df['Kemenag - Google'] < 0.5)]
maximum_difference

,Unnamed: 0,Chapter,Verse,Google Translate,Kemenag,King Fahd,Fahd - Google,Kemenag - Google,Fahd - Kemenag
337,337,3,54,"Dan mereka berencana, dan Allah pun berencana. Dan Allah adalah sebaik-baik perencana.","Dan mereka (orang-orang kafir) membuat tipu daya, maka Allah pun membalas tipu daya. Dan Allah sebaik-baik pembalas tipu daya.","Orang-orang kafir itu membuat tipu daya, dan Allah membalas tipu daya mereka itu. Dan Allah sebaik-baik pembalas tipu daya.",0.447102,0.476180,0.945183
816,816,9,37,"Interkalasi hanyalah penambahan kekufuran yang dengannya orang-orang kafir disesatkan. Mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain agar mereka dapat menyempurnakan apa yang diharamkan Allah, dan dengan demikia...","Sesungguhnya pengunduran (bulan haram) itu hanya menambah kekafiran. Orang-orang kafir disesatkan dengan (pengunduran) itu, mereka menghalalkannya satu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menyesuaikan dengan bilangan...","Sesungguhnya mengundur-undurkan bulan haram itu adalah menambah kekafiran, disesatkan orang-orang yang kafir dengan mengundur-undurkan itu, mereka menghalalkannya pada suatu tahun dan mengharamkannya pada tahun yang lain, agar mereka dapat menses...",0.346780,0.354384,0.879055
986,986,33,14,"Dan jika surga itu datang kepada mereka dari daerahnya, kemudian mereka diminta untuk mengujinya, niscaya mereka akan mengujinya, dan mereka tidak akan tinggal di dalamnya kecuali sebentar saja.","Dan kalau (Yasrib) diserang dari segala penjuru, dan mereka diminta agar membuat kekacauan, niscaya mereka mengerjakannya; dan hanya sebentar saja mereka menunggu.","Kalau (Yaṡrib) diserang dari segala penjuru, kemudian diminta kepada mereka supaya murtad , niscaya mereka mengerjakannya dan mereka tiada akan menunda untuk murtad itu, melainkan dalam waktu yang singkat.",0.438173,0.428026,0.772614


In [34]:
df_mean_fahd_google = df.groupby('Chapter')['Fahd - Google'].mean().reset_index()
df_mean_kemenag_google = df.groupby('Chapter')['Kemenag - Google'].mean().reset_index()


df_std_fahd_google = df.groupby('Chapter')['Fahd - Google'].std().reset_index()
df_std_kemenag_google = df.groupby('Chapter')['Kemenag - Google'].std().reset_index()


In [36]:
df_mean_fahd_google

,Chapter,Fahd - Google
0,2,0.820412
1,3,0.823510
2,4,0.818048
3,5,0.832971
4,9,0.807629
5,24,0.801286
6,33,0.829647
7,47,0.837569
8,48,0.807638
9,49,0.851366


In [37]:
df_mean_kemenag_google

,Chapter,Kemenag - Google
0,2,0.836795
1,3,0.844923
2,4,0.825697
3,5,0.833616
4,9,0.823944
5,24,0.823533
6,33,0.831098
7,47,0.826737
8,48,0.811333
9,49,0.837540


In [38]:
df_std_fahd_google

,Chapter,Fahd - Google
0,2,0.097392
1,3,0.107465
2,4,0.093525
3,5,0.090328
4,9,0.107501
5,24,0.114689
6,33,0.100417
7,47,0.090149
8,48,0.130074
9,49,0.118871


In [39]:
df_std_kemenag_google

,Chapter,Kemenag - Google
0,2,0.087024
1,3,0.086941
2,4,0.087390
3,5,0.086576
4,9,0.102026
5,24,0.088391
6,33,0.095863
7,47,0.099716
8,48,0.121718
9,49,0.108646


In [41]:
df_std_fahd_google['Fahd - Google'].mean()

0.09271860688492285

In [42]:
df_std_kemenag_google['Kemenag - Google'].mean()

0.08614932913816385

In [45]:
from keybert import KeyBERT
kw_model = KeyBERT(model = 'LazarusNLP/all-indo-e5-small-v4')